# 020 — Training

Train UNet, ResUNet, Attention UNet, and EfficientNet UNet on the RGB → IR translation task.
Checkpoints are saved to `models/{arch}/best_model.keras`.
Logs are written to `logs/{arch}/` for TensorBoard.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    grouped_train_val_test_split,
    load_image_pairs,
)
from scripts.reproducibility import set_global_seed
from scripts.trainer import compile_model, get_callbacks, get_model
from scripts.visualization import plot_training_curves

# Seed Python / NumPy / TensorFlow from settings.SEED for reproducible runs.
set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Build datasets

In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = grouped_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    seed=settings.SEED,
)

# crop_size only affects the augmented training split; val stays full-image.
train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Loss functions

The loss is never passed by hand: it is resolved from the architecture through the
`_ARCH_LOSSES` registry in `scripts/trainer.py` (`get_loss_name` → `build_loss`), so
training and checkpoint reloading can never disagree.

| Architecture | Loss |
|---|---|
| UNet, ResUNet, Attention UNet | `combined_loss` — MAE + (1 − SSIM) |
| EfficientNet UNet | `combined_loss_advanced` — MAE + Laplacian pyramid + FFT |
| *(unassigned)* | `combined_loss_normalized` — MAE + per-window z-score |

The **Laplacian pyramid** loss decomposes prediction error into spatial frequency bands
and weights finer detail more heavily (geometric sequence: level 0 weight = `2^(N-1)`, halved each level).
The **FFT loss** penalises magnitude-spectrum errors uniformly across all frequencies,
preventing the model from sacrificing high-frequency accuracy to reduce low-frequency error.
Both terms target the frequency band where underdrawing strokes live.

`combined_loss_normalized` implements the local-normalization idea from `note.md` §2: it
measures error in units of **local contrast** rather than absolute gray level, which is the
metric the delta is actually read in (`040_inference.ipynb` passes it through CLAHE). Two
things to know before enabling it on an architecture:

- Its z-score term is invariant to local affine gray-level changes, so the MAE anchor is
  mandatory — without it the absolute gray level is unconstrained and the raw delta loses
  its meaning.
- The two terms are not on a comparable scale: `mean|Δz|` measures ~9× the MAE of the same
  prediction on held-out paintings, so `NORM_LOSS_BETA` (0.03) is deliberately far below
  `NORM_LOSS_ALPHA` (1.0). `ZSCORE_SIGMA_FLOOR` (0.01) is the 10th percentile of the local
  standard deviation measured over the IR set, and bounds the gradient in flat regions.

To train an architecture with it, change its entry in `_ARCH_LOSSES` to
`LossName.NORMALIZED`; the weights come from `settings` and are `.env`-overridable.

The cell below visualises the pyramid decomposition and FFT spectra of one training sample.

In [ ]:
import numpy as np
import tensorflow as tf
from PIL import Image

# --- Laplacian pyramid on a sample IR image ---
ir_np = np.array(Image.open(train_pairs[0][1]).convert("L")).astype(np.float32) / 255.0
ir_t = tf.constant(ir_np[np.newaxis, ..., np.newaxis])  # (1, H, W, 1)


def _lap_level(x: tf.Tensor) -> tuple:
    low = tf.nn.avg_pool2d(x, ksize=2, strides=2, padding="VALID")
    up = tf.image.resize(low, tf.shape(x)[1:3], method="bilinear")
    detail = (x - up)[0, ..., 0].numpy()
    return detail, low


LEVELS = 5
details, x = [], ir_t
for _ in range(LEVELS):
    d, x = _lap_level(x)
    details.append(d)

fig, axes = plt.subplots(1, LEVELS + 1, figsize=(4 * (LEVELS + 1), 4))
fig.suptitle("Laplacian pyramid — sample IR image  (level 0 = finest detail)")
axes[0].imshow(ir_np, cmap="gray")
axes[0].set_title("Original IR")
axes[0].axis("off")
for i, d in enumerate(details):
    d_disp = (d - d.min()) / (d.max() - d.min() + 1e-8)
    axes[i + 1].imshow(d_disp, cmap="RdBu_r")
    axes[i + 1].set_title(f"Level {i}")
    axes[i + 1].axis("off")
plt.tight_layout()
plt.show()

# --- FFT magnitude spectra ---
rgb_np = (
    np.array(Image.open(train_pairs[0][0]).convert("RGB")).astype(np.float32) / 255.0
)
log_rgb = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(rgb_np[..., 0]))))
log_ir = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(ir_np))))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("FFT magnitude spectra (log scale) — same patch")
axes[0].imshow(log_rgb, cmap="inferno")
axes[0].set_title("RGB (channel R)")
axes[0].axis("off")
axes[1].imshow(log_ir, cmap="inferno")
axes[1].set_title("IR")
axes[1].axis("off")
axes[2].imshow(np.abs(log_rgb - log_ir), cmap="hot")
axes[2].set_title("Spectral difference |R − IR|")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 3. Train all architectures

UNet, ResUNet, and Attention UNet use `combined_loss` (MAE + SSIM).
EfficientNet UNet uses `combined_loss_advanced` (MAE + Laplacian pyramid + FFT),
which better preserves the high-frequency detail that a pretrained encoder can reproduce.

Set `EPOCHS = 2` for a quick smoke test.

In [ ]:
ARCHS = ["efficientnet_unet"]
EPOCHS = settings.EPOCHS  # -- change to settings.EPOCHS for full training

histories: dict = {}

for arch in ARCHS:
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {arch}")
    print(f"{'=' * 60}")

    model = get_model(arch)
    model = compile_model(
        model,
        arch,
        lr=settings.LEARNING_RATE,
        loss_alpha=settings.LOSS_ALPHA,
    )
    model.summary(line_length=80)

    callbacks = get_callbacks(
        arch,
        log_dir=settings.LOGS_DIR,
        model_dir=settings.MODELS_DIR,
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    histories[arch] = history.history

    best_val_loss = min(history.history["val_loss"])
    print(f"\nBest val_loss ({arch}): {best_val_loss:.4f}")

In [ ]:
ARCHS = ["unet", "resunet", "attention_unet", "efficientnet_unet"]
EPOCHS = settings.EPOCHS  # -- change to settings.EPOCHS for full training

histories: dict = {}

for arch in ARCHS:
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {arch}")
    print(f"{'=' * 60}")

    model = get_model(arch)
    model = compile_model(
        model,
        arch,
        lr=settings.LEARNING_RATE,
        loss_alpha=settings.LOSS_ALPHA,
    )
    model.summary(line_length=80)

    callbacks = get_callbacks(
        arch,
        log_dir=settings.LOGS_DIR,
        model_dir=settings.MODELS_DIR,
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    histories[arch] = history.history

    best_val_loss = min(history.history["val_loss"])
    print(f"\nBest val_loss ({arch}): {best_val_loss:.4f}")

## 3. Training curves

In [ ]:
for arch, history in histories.items():
    fig = plot_training_curves(history, title=f"Training history — {arch}")
    plt.show()

## 4. Summary

Checkpoints saved to `models/{arch}/best_model.keras`.  
Run `tensorboard --logdir logs/` to inspect curves interactively.

> **Note:** `efficientnet_unet` downloads EfficientNetB0 ImageNet weights on first use.
> Subsequent runs use the local Keras cache.

In [ ]:
for arch in ARCHS:
    ckpt = settings.MODELS_DIR / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25}: {status}  ({ckpt})")